# DrugRadar — Notebook 02: Train the Model**Run this AFTER Notebook 01.**
This trains an XLM-RoBERTa model to detect Adverse Drug Events.
>  This notebook takes **35–45 minutes** to finish.> Once you start Cell 5 (training), just leave it running — do not close the tab.
Run every cell top to bottom. When you see  at the end, open Notebook 03.

## Cell 1 — Imports and settings

In [1]:
from google.colab import drivedrive.mount('/content/drive')
import osimport pandas as pd
# Load from Google Drivetrain_df = pd.read_csv("/content/drive/MyDrive/DrugRadar/data/raw/train.csv")test_df  = pd.read_csv("/content/drive/MyDrive/DrugRadar/data/raw/test.csv")
print(f"Train rows : {len(train_df)}")print(f"Test rows  : {len(test_df)}")

Mounted at /content/drive
✅ Loaded from Google Drive!
Train rows : 18812
Test rows  : 4704


In [2]:
import os, jsonimport numpy as npimport pandas as pdimport torchfrom torch import nnfrom torch.utils.data import Datasetfrom transformers import (AutoTokenizer,AutoModelForSequenceClassification,TrainingArguments,Trainer,EarlyStoppingCallback,)from sklearn.metrics import classification_report, f1_scorefrom sklearn.utils.class_weight import compute_class_weight
# ── Settings ──────────────────────────────────────────────────────────────MODEL_NAME = "xlm-roberta-base"MAX_LEN    = 128BATCH_SIZE = 16EPOCHS     = 5LR         = 2e-5SEED       = 42OUTPUT_DIR = "models/checkpoints/xlmr_ade"DEVICE     = "cuda" if torch.cuda.is_available() else "cpu"
os.makedirs(OUTPUT_DIR, exist_ok=True)torch.manual_seed(SEED)np.random.seed(SEED)
print(f"Device : {DEVICE}")print(f"Model  : {MODEL_NAME}")if DEVICE == "cpu":else:

Device : cuda
Model  : xlm-roberta-base
✅ GPU ready!


## Cell 2 — Load the train and test data

In [4]:
# Data already loaded from Google Drive in Cell 1 — just confirming hereprint(f"Train rows : {len(train_df)}")print(f"Test rows  : {len(test_df)}")print(f"Columns    : {list(train_df.columns)}")print(f"\nTrain ADE rate : {train_df['label'].mean()*100:.1f}%")print(f"Test  ADE rate : {test_df['label'].mean()*100:.1f}%")
train_texts  = train_df["text"].tolist()train_labels = train_df["label"].tolist()test_texts   = test_df["text"].tolist()test_labels  = test_df["label"].tolist()


Train rows : 18812
Test rows  : 4704
Columns    : ['text', 'label']

Train ADE rate : 29.0%
Test  ADE rate : 29.0%

✅ Data loaded!


## Cell 3 — Define the dataset class and weighted trainer> You do not need to understand this code. Just run it.

In [5]:
class ADEDataset(Dataset):def __init__(self, texts, labels, tokenizer):self.texts = textsself.labels = labelsself.tokenizer = tokenizer
def __len__(self):return len(self.texts)
def __getitem__(self, idx):enc = self.tokenizer(str(self.texts[idx]),max_length=MAX_LEN,padding="max_length",truncation=True,return_tensors="pt",)return {"input_ids":      enc["input_ids"].squeeze(),"attention_mask": enc["attention_mask"].squeeze(),"labels":         torch.tensor(int(self.labels[idx]), dtype=torch.long),}

class WeightedTrainer(Trainer):"""Uses class weights so the model doesn't ignore the minority class."""def __init__(self, class_weights, *args, **kwargs):super().__init__(*args, **kwargs)self.class_weights = class_weights.to(DEVICE)
def compute_loss(self, model, inputs, return_outputs=False, **kwargs):labels = inputs.pop("labels")outputs = model(**inputs)loss = nn.CrossEntropyLoss(weight=self.class_weights)(outputs.logits, labels)return (loss, outputs) if return_outputs else loss

def compute_metrics(eval_pred):preds = np.argmax(eval_pred.predictions, axis=-1)return {"macro_f1":    f1_score(eval_pred.label_ids, preds, average="macro"),"positive_f1": f1_score(eval_pred.label_ids, preds, average="binary", pos_label=1),}

✅ Classes defined!


## Cell 4 — Load the tokenizer and model> This downloads the model (~1GB). Takes 2-3 minutes.

In [6]:
print(f"Loading tokenizer: {MODEL_NAME} ...")tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print(f"\nLoading model: {MODEL_NAME} ...")model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2).to(DEVICE)
# Build datasetstrain_dataset = ADEDataset(train_texts, train_labels, tokenizer)eval_dataset  = ADEDataset(test_texts,  test_labels,  tokenizer)
# Compute class weights (handles imbalance)classes = sorted(set(train_labels))weights = compute_class_weight("balanced", classes=np.array(classes), y=np.array(train_labels))class_weights = torch.tensor(weights, dtype=torch.float)print(f"\nClass weights: No ADE={weights[0]:.2f}, ADE={weights[1]:.2f}")

Loading tokenizer: xlm-roberta-base ...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


✅ Tokenizer loaded!

Loading model: xlm-roberta-base ...


XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
classifier.out_proj.bias    | MISSING    | 
classifier.dense.weight     | MISSING    | 
classifier.out_proj.weight  | MISSING    | 
classifier.dense.bias       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


✅ Model loaded!

Class weights: No ADE=0.70, ADE=1.72
(Higher weight = model pays more attention to that class)


## Cell 5 — Train the model> This takes **35–45 minutes**. Leave the tab open. You will see progress printed below.

In [7]:
training_args = TrainingArguments(output_dir=OUTPUT_DIR,num_train_epochs=EPOCHS,per_device_train_batch_size=BATCH_SIZE,per_device_eval_batch_size=BATCH_SIZE,learning_rate=LR,weight_decay=0.01,warmup_ratio=0.1,eval_strategy="epoch",save_strategy="epoch",load_best_model_at_end=True,metric_for_best_model="macro_f1",greater_is_better=True,logging_steps=50,fp16=(DEVICE == "cuda"),seed=SEED,report_to="none",)
trainer = WeightedTrainer(class_weights=class_weights,model=model,args=training_args,train_dataset=train_dataset,eval_dataset=eval_dataset,compute_metrics=compute_metrics,callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],)
trainer.train()

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Starting training... grab a coffee ☕
You will see loss numbers printing every 50 steps.



Epoch,Training Loss,Validation Loss,Macro F1,Positive F1
1,0.251155,0.206911,0.916442,0.883509
2,0.229476,0.231967,0.934672,0.907980
3,0.192506,0.215917,0.939692,0.915756
4,0.127345,0.264805,0.936878,0.911785
5,0.128227,0.295192,0.940872,0.917052


There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye


✅ Training complete!


## Cell 6 — Save the model

In [8]:
trainer.save_model(OUTPUT_DIR)tokenizer.save_pretrained(OUTPUT_DIR)

✅ Model saved to: models/checkpoints/xlmr_ade


## Cell 7 — Evaluate and save predictions

In [9]:
preds_out = trainer.predict(eval_dataset)preds     = np.argmax(preds_out.predictions, axis=-1)
print("── Final Results ──────────────────────────────────")print(classification_report(test_labels, preds, target_names=["No ADE", "ADE"]))
# Save predictions CSV (used by Notebook 03)pred_df = pd.DataFrame({"text":       test_texts,"true_label": test_labels,"pred_label": preds,"confidence": preds_out.predictions.max(axis=-1).tolist(),})pred_df.to_csv("/content/drive/MyDrive/DrugRadar/outputs/predictions/baseline_preds.csv", index=False)
# Save false negatives (ADE posts the model missed) → used by Notebook 03fn = pred_df[(pred_df["true_label"] == 1) & (pred_df["pred_label"] == 0)]
# Update this path to your Drive:fn.to_csv("/content/drive/MyDrive/DrugRadar/data/processed/false_negatives.csv", index=False)


── Final Results ──────────────────────────────────
              precision    recall  f1-score   support

      No ADE       0.98      0.95      0.96      3340
         ADE       0.89      0.94      0.92      1364

    accuracy                           0.95      4704
   macro avg       0.93      0.95      0.94      4704
weighted avg       0.95      0.95      0.95      4704

✅ Saved: /content/drive/MyDrive/DrugRadar/outputs/predictions/baseline_preds.csv
✅ Saved: /content/drive/MyDrive/DrugRadar/data/processed/false_negatives.csv (76 false negatives)

✅ All done! Open Notebook 03 next.
